# 05 — Methodological Audit: Test-Set Discipline Correction

**Why this notebook exists.** In `04_feature_selection_and_shap.ipynb`,
permutation importance and SHAP were computed on the **held-out test split**
(`test_df`), and those results (together with a trim-confirmation step that
also printed test-set numbers) directly motivated dropping `MaxHR` and
`HDL` from the Enhanced feature set. This means the test set was inspected
**more than once** before the final feature set was frozen — not "used
once for final evaluation" as the surrounding reports claimed. This
notebook:

1. Confirms exactly what happened (grep-level audit of notebook 04).
2. **Redoes the MaxHR/HDL decision using ONLY the 80% training split**, with
   leakage-safe, out-of-fold 5-fold CV — `test_df` is never touched below.
3. Runs a **paired statistical test** (Random Forest vs XGBoost) to settle
   the "statistically equivalent" wording question precisely, rather than
   asserting it from eyeballing standard deviations.

The original notebook 04 and its outputs are **not modified** — this is an
additive audit, per instruction not to hide or discard existing results.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from scipy import stats

RANDOM_STATE = 42
TARGET = 'Heart Disease'
pd.set_option('display.width', 160)

train_df = pd.read_pickle("train.pkl")
test_df = pd.read_pickle("test.pkl")
print("Train:", train_df.shape, "Test:", test_df.shape)
print("NOTE: test_df is loaded for provenance only - watch that it is never")
print("passed to any .fit(), permutation_importance(), or scoring call below.")

Train: (828, 30) Test: (207, 30)
NOTE: test_df is loaded for provenance only - watch that it is never
passed to any .fit(), permutation_importance(), or scoring call below.


## 1. Confirm exactly what notebook 04 did (grep-level check)

Re-running this here (rather than trusting memory) so the audit itself is
verifiable.

In [2]:
nb04 = json.load(open("04_feature_selection_and_shap.ipynb", encoding="utf-8"))
for i, c in enumerate(nb04['cells']):
    if c['cell_type'] == 'code':
        src = ''.join(c['source'])
        if 'test_df' in src:
            print(f"cell {i} references test_df:")
            for line in src.split('\n'):
                if 'test_df' in line:
                    print("   ", line.strip())

cell 1 references test_df:
    test_df = pd.read_pickle("test.pkl")
cell 8 references test_df:
    result = permutation_importance(pipe, test_df, test_df[TARGET], n_repeats=30,
cell 11 references test_df:
    X_test_transformed = pre.transform(test_df)
cell 14 references test_df:
    proba = pipe.predict_proba(test_df)[:, 1]
    pred = pipe.predict(test_df)
    print(f"{name:40s} CV-AUC={cv_scores.mean():.4f}+/-{cv_scores.std():.4f}  Test-AUC={roc_auc_score(test_df[TARGET],proba):.4f}  Test-Recall={recall_score(test_df[TARGET],pred):.4f}")
cell 17 references test_df:
    proba = pipe.predict_proba(test_df)[:, 1]
    pred = pipe.predict(test_df)
    y = test_df[TARGET]
    print(f"=== {name} (XGBoost) - HELD-OUT TEST (n={len(test_df)}) ===")


**Confirmed:** cells 8 (permutation importance) and 11 (SHAP) compute
importance directly on `test_df`, and cell 14 (the full-vs-trimmed
comparison) prints test-set AUC/recall alongside the CV numbers as part of
the same decision-making cell — all before cell 17's "final" evaluation.
The test set was genuinely inspected multiple times, not once. This is a
real process violation, not a false alarm.

## 2. Redo: CV-only, out-of-fold permutation importance (train_df ONLY)

For each of the 5 stratified folds: fit on that fold's training portion,
compute permutation importance on that fold's **validation** portion (data
the model did not see during that fit) — average across folds. `test_df` is
never referenced in this cell.

In [3]:
BASIC_NUMERIC = ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)']
BASIC_BINARY = ['Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain']
BASIC_CATEGORICAL = ['Sex']
ENHANCED_NUMERIC_ADD = ['Total_Cholesterol(mg/dL)', 'HDL(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)', 'MaxHR']
ENHANCED_FULL = BASIC_NUMERIC + ENHANCED_NUMERIC_ADD
ENHANCED_TRIMMED = BASIC_NUMERIC + ['Total_Cholesterol(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)']

def make_pipeline(numeric_cols, algo):
    numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())])
    binary_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent'))])
    cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
    pre = ColumnTransformer([('num', numeric_pipe, numeric_cols), ('bin', binary_pipe, BASIC_BINARY), ('cat', cat_pipe, BASIC_CATEGORICAL)])
    spw = (train_df[TARGET]==0).sum() / (train_df[TARGET]==1).sum()
    if algo == 'xgb':
        model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', scale_pos_weight=spw,
                               n_estimators=200, max_depth=3, learning_rate=0.05, n_jobs=-1)
    else:
        model = RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced',
                                        n_estimators=400, max_depth=6, max_features='sqrt', n_jobs=-1)
    return Pipeline([('preprocess', pre), ('model', model)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X, y = train_df.reset_index(drop=True), train_df[TARGET].reset_index(drop=True)

fold_importances = []
for tr_idx, val_idx in cv.split(X, y):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    pipe = make_pipeline(ENHANCED_FULL, 'xgb')
    pipe.fit(X_tr, y_tr)  # fit on fold-train only
    result = permutation_importance(pipe, X_val, y_val, n_repeats=30,  # evaluated on fold-VALIDATION only
                                     random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1)
    used_cols = ENHANCED_FULL + BASIC_BINARY + BASIC_CATEGORICAL
    feature_names = list(pipe.named_steps['preprocess'].feature_names_in_)
    imp = pd.Series(result.importances_mean, index=feature_names)
    fold_importances.append(imp.loc[imp.index.isin(used_cols)])

mean_importance = pd.DataFrame(fold_importances).mean().sort_values(ascending=False)
mean_importance.round(4)

LDL(mg/dL)                  0.0565
Triglycerides(mg/dL)        0.0293
RBS(mmol/L)                 0.0193
Total_Cholesterol(mg/dL)    0.0178
BP(mmHg)                    0.0079
Diabetes                    0.0064
H/O ChestPain               0.0040
Age                         0.0035
Hypertension                0.0022
Weight (kg)                 0.0014
MaxHR                       0.0005
HDL(mg/dL)                  0.0002
Height (cm)                 0.0002
Family H/O                  0.0001
Sex                         0.0000
dtype: float64

**MaxHR (0.0005) and HDL (0.0002) remain the two weakest features by a
wide margin** — same conclusion as the (methodologically flawed) test-set
version, now derived cleanly.

## 3. Redo: full-vs-trimmed comparison, CV-only (decision frozen here)

In [4]:
for name, cols in [('Enhanced_full (6 fields)', ENHANCED_FULL), ('Enhanced_trimmed (4 fields)', ENHANCED_TRIMMED)]:
    pipe = make_pipeline(cols, 'xgb')
    auc_scores = cross_val_score(pipe, train_df, train_df[TARGET], cv=cv, scoring='roc_auc')
    recall_scores = cross_val_score(pipe, train_df, train_df[TARGET], cv=cv, scoring='recall')
    print(f"{name:30s} CV-AUC={auc_scores.mean():.4f}+/-{auc_scores.std():.4f}  CV-Recall={recall_scores.mean():.4f}+/-{recall_scores.std():.4f}")
print("\nDecision: drop MaxHR and HDL. Frozen from CV-only evidence above - test_df not consulted.")

Enhanced_full (6 fields)       CV-AUC=0.9817+/-0.0045  CV-Recall=0.9295+/-0.0248


Enhanced_trimmed (4 fields)    CV-AUC=0.9815+/-0.0044  CV-Recall=0.9359+/-0.0151

Decision: drop MaxHR and HDL. Frozen from CV-only evidence above - test_df not consulted.


No CV-AUC cost (0.9817 → 0.9815) and CV-recall actually improves
slightly (0.9295 → 0.9359) with the trimmed set. **Same conclusion as
before, now on clean grounds — the Enhanced feature list does not change.**

## 4. Paired significance test: Random Forest vs XGBoost (frozen feature sets, train_df only)

Settles the "statistically equivalent" wording question with an actual
test rather than an eyeballed standard deviation. The same `cv` object
(fixed `random_state`) produces identical fold splits across both
algorithms, so fold scores are legitimately paired.

In [5]:
for fs_name, cols in [('A_Basic', BASIC_NUMERIC), ('B_Enhanced_trimmed', ENHANCED_TRIMMED)]:
    xgb_scores = cross_val_score(make_pipeline(cols, 'xgb'), train_df, train_df[TARGET], cv=cv, scoring='roc_auc')
    rf_scores = cross_val_score(make_pipeline(cols, 'rf'), train_df, train_df[TARGET], cv=cv, scoring='roc_auc')
    t_stat, p_value = stats.ttest_rel(xgb_scores, rf_scores)
    print(f"{fs_name}: XGBoost={np.round(xgb_scores,4)}  RF={np.round(rf_scores,4)}")
    print(f"  paired t-test: t={t_stat:.3f}, p={p_value:.4f}\n")

A_Basic: XGBoost=[0.9003 0.9112 0.9366 0.9679 0.9328]  RF=[0.8989 0.8972 0.9173 0.9658 0.9355]
  paired t-test: t=1.627, p=0.1792



B_Enhanced_trimmed: XGBoost=[0.9774 0.9821 0.9759 0.9878 0.9843]  RF=[0.974  0.983  0.9789 0.9913 0.9849]
  paired t-test: t=-0.755, p=0.4921



**No statistically significant difference found** (p=0.18 Basic,
p=0.49 Enhanced). With only 5 paired folds this test has limited power —
read as "no evidence of a difference," not proof of true equivalence.
Reports now say "comparable performance" rather than "statistically
equivalent" for this reason.

## 5. Confusion-matrix / metric internal-consistency check

Re-derives accuracy/precision/recall/specificity/F1 directly from the
reported confusion matrices to confirm no arithmetic errors slipped into
the written reports.

In [6]:
def check(name, tn, fp, fn, tp, reported):
    total = tn + fp + fn + tp
    acc = (tn + tp) / total
    prec = tp / (tp + fp)
    rec = tp / (tp + fn)
    spec = tn / (tn + fp)
    f1 = 2 * prec * rec / (prec + rec)
    computed = {'accuracy': acc, 'precision': prec, 'recall': rec, 'specificity': spec, 'f1': f1}
    print(f"{name}: n={total}")
    ok = True
    for k, v in reported.items():
        match = abs(computed[k] - v) < 0.001
        ok &= match
        print(f"  {k:12s} reported={v:.4f}  recomputed={computed[k]:.4f}  {'OK' if match else 'MISMATCH'}")
    print(f"  -> {'CONSISTENT' if ok else 'INCONSISTENT'}\n")

check("A_Basic / XGBoost (final)", tn=78, fp=12, fn=20, tp=97,
      reported={'accuracy':0.8454,'precision':0.8899,'recall':0.8291,'specificity':0.8667,'f1':0.8584})
check("B_Enhanced_trimmed / XGBoost (final)", tn=86, fp=4, fn=10, tp=107,
      reported={'accuracy':0.9324,'precision':0.9640,'recall':0.9145,'specificity':0.9556,'f1':0.9386})
check("B_full / RandomForest", tn=85, fp=5, fn=16, tp=101,
      reported={'accuracy':0.8986,'precision':0.9528,'recall':0.8632,'specificity':0.9444,'f1':0.9058})

A_Basic / XGBoost (final): n=207
  accuracy     reported=0.8454  recomputed=0.8454  OK
  precision    reported=0.8899  recomputed=0.8899  OK
  recall       reported=0.8291  recomputed=0.8291  OK
  specificity  reported=0.8667  recomputed=0.8667  OK
  f1           reported=0.8584  recomputed=0.8584  OK
  -> CONSISTENT

B_Enhanced_trimmed / XGBoost (final): n=207
  accuracy     reported=0.9324  recomputed=0.9324  OK
  precision    reported=0.9640  recomputed=0.9640  OK
  recall       reported=0.9145  recomputed=0.9145  OK
  specificity  reported=0.9556  recomputed=0.9556  OK
  f1           reported=0.9386  recomputed=0.9386  OK
  -> CONSISTENT

B_full / RandomForest: n=207
  accuracy     reported=0.8986  recomputed=0.8986  OK
  precision    reported=0.9528  recomputed=0.9528  OK
  recall       reported=0.8632  recomputed=0.8632  OK
  specificity  reported=0.9444  recomputed=0.9444  OK
  f1           reported=0.9058  recomputed=0.9058  OK
  -> CONSISTENT



## 6. Conclusion

- The MaxHR/HDL trim decision is **re-confirmed** using clean, leakage-safe,
  CV-only evidence — the final Enhanced feature list is unchanged.
- Random Forest vs XGBoost: **no statistically significant difference**
  (paired t-test) — wording corrected to "comparable" throughout the
  reports.
- All checked confusion matrices are internally consistent with their
  reported metrics.
- See `../reports/ml_final_audit.md` for the full written audit and the
  answers to questions A–F.